# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">4주차 · 사람이 정한 규칙을 지우고 데이터가 찾은 취향으로 추천하기</mark>

지난 세 주 동안 추천의 규칙은 **사람이 정했습니다.** 주력 섹터로 나누자, 평균 위험도로 나누자 — 강사가 고른 것입니다.

오늘은 **아무도 정하지 않습니다.** 거래 기록만 주고 **모델이 스스로 찾게** 합니다. 그 방법이 **행렬분해**입니다.

---

### 오늘의 구성

| 파트 | 종류 | 어디서 | 하는 일 |
|---|---|---|---|
| 1 | 개념 | 슬라이드 | 큰 표를 작은 표 둘로 쪼개면 빈칸이 채워진다 |
| 2 | 실습 | **이 노트북 4.2~4.8** | 직접 쪼개고, 추천을 만들고, 요인 개수를 바꿔 본다 |
| 3 | 마무리 | 슬라이드 | 오늘의 용어 · 점수판 · 다음 주 |

이 노트북은 **실습 파트**입니다. 그래서 절 번호가 `4.` 부터 시작합니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

이 노트북은 **아무것도 설치하지 않고** 브라우저에서 바로 실행할 수 있습니다.

### Google Colab 으로 열기

1. 아래 주소를 눌러 주세요.
   - https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/04_matrix_factorization.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 "실습 준비" 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다
import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4 · 실습 — 모델에게 취향을 찾게 한다</mark>

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 오늘 나오는 말  `[PPT]`</mark>

1. **행렬 (matrix)** — 가로 세로로 칸이 있는 **표**
   - 오늘은 **사람 285 × 종목 100** 짜리 표를 만듭니다. 담았으면 1, 안 담았으면 0.
2. **행렬분해 (matrix factorization)** — 큰 표 하나를 **작은 표 두 개로 쪼개는 것**
   - 코드에서는 `TruncatedSVD` 라고 적습니다. 오늘 수업 전체가 이 이야기입니다.
3. **잠재 요인 (latent factor)** — 사람과 종목을 설명하는 **축**
   - 오늘은 8개로 씁니다. **이름은 아무도 안 붙여 줍니다.** 그래서 「잠재」입니다.
4. **과적합 (overfitting)** — 본 것은 다 맞히는데 **새 것은 못 맞히는** 상태
   - 요인을 너무 많이 주면 이렇게 됩니다. 4.6 에서 직접 확인합니다.
5. **콜드 스타트 (cold start)** — 지난주 복습. 기록이 없어 취향을 짐작할 재료가 없는 상태
   - 오늘도 이 사람들은 못 풉니다. 4.7 에서 확인합니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 · 데이터와 채점 방법 (지난주 그대로)</mark>

**여기서 하는 일** — 데이터를 읽고, **3주차의 채점 방법을 그대로** 가져옵니다.

**1. 왜 채점을 그대로 쓰나**
- 오늘 바꾸는 것은 **추천을 만드는 방법 하나**입니다. 재는 자가 같이 바뀌면 무엇 때문에 점수가 변했는지 알 수 없습니다.
- 시간순 분할, 이미 담은 것 빼기 — 전부 지난주와 똑같습니다.

**2. 지난주와 딱 하나 다른 점**
- 지난주 채점 함수는 **세그먼트 방식이 안에 박혀 있었습니다.**
- 오늘은 **추천을 만드는 함수를 밖에서 받도록** 바꿉니다. 그래야 오늘 만들 방식으로도 채점할 수 있습니다.

**3. 설명 없이 한 번에 실행합니다** — 지난주에 이미 한 것이라 결과만 확인합니다.

In [2]:
# 이 셀에서 하는 일 — 오늘 쓸 도구를 불러온다
# 왜 하나 — pandas 로 표를 다루고, numpy 로 숫자 묶음을 다루고, recsys.py 에는 1~3주차 함수가 있다
import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구

sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 칸 안쪽 폴더도 찾게 한다

import pandas as pd                         # 표를 다루는 도구. 앞으로 pd 라고 부른다
import numpy as np                          # 숫자 묶음을 빠르게 다루는 도구. np 라고 부른다
import recsys                               # 이 수업용으로 만든 도구 모음

print("도구 준비 완료 · pandas", pd.__version__, "· numpy", np.__version__)   # 버전까지 찍어 두면 나중에 결과가 다를 때 비교하기 좋다

도구 준비 완료 · pandas 3.0.5 · numpy 2.5.2


In [3]:
# 이 셀에서 하는 일 — 데이터를 읽고, 3주차와 똑같이 학습 구간·채점 구간으로 나눈다
# 왜 하나 — 잣대가 지난주와 같아야 0.2684 와 견줄 수 있다
items, users, interactions = recsys.load()          # 종목·투자자·거래 기록 세 표
print(f"종목 {len(items)}개 · 투자자 {len(users)}명 · 거래 기록 {len(interactions):,}건")   # 1~3주차와 같은 데이터인지 확인

train, test, 기준시점 = recsys.split_by_time(interactions)   # 1주차부터 쓰던 시간순 분할
print(f"학습 구간 {len(train):,}건 · 채점 구간 {len(test):,}건 · 자른 날짜 {기준시점.date()}")   # 3주차와 같은 7,095 / 991

이름_사전 = items.set_index("item_id")["name"].to_dict()      # {종목 번호: 이름}
위험도_사전 = items.set_index("item_id")["risk_level"].to_dict()   # {종목 번호: 1~5}
print(f"사전 2개 준비 · I079 → {이름_사전['I079']} · 위험도 {위험도_사전['I079']}")

종목 100개 · 투자자 300명 · 거래 기록 8,086건
학습 구간 7,095건 · 채점 구간 991건 · 자른 날짜 2026-07-30
사전 2개 준비 · I079 → KODEX 레버리지 · 위험도 5


In [4]:
# 이 셀에서 하는 일 — 3주차 채점 방법을 가져온다. 추천 만드는 함수만 밖에서 받는다
# 왜 하나 — 오늘은 추천 만드는 방법이 바뀐다. 채점하는 부분은 한 글자도 안 바꾼다
이미_담은것 = train.groupby("user_id")["item_id"].apply(set).to_dict()   # {사람: 학습 구간에 담은 집합}
전체_인기순위 = list(train["item_id"].value_counts().index)              # 많이 담긴 순 (기록 없는 사람용)


def 채점하기(추천하기):
    """추천 함수를 받아 사람별 Recall@10 을 돌려준다. 3주차와 재는 방식이 같다."""
    사람별_점수 = {}                                  # {사람: 그 사람의 Recall@10}
    for 사람, 그_사람의_기록 in test.groupby("user_id"):   # 채점 대상을 한 명씩
        이미 = 이미_담은것.get(사람, set())             # 학습 구간에 이미 담은 것
        정답 = set(그_사람의_기록["item_id"]) - 이미      # 이미 담은 것은 정답에서 뺀다
        if len(정답) == 0:                            # 맞힐 것이 없는 사람은
            continue                                  # 채점에서 뺀다
        추천 = 추천하기(사람)                           # 밖에서 받은 함수로 10개를 받는다
        사람별_점수[사람] = len(정답 & set(추천)) / len(정답)   # 맞힌 개수 ÷ 정답 개수
    return 사람별_점수


def 평균내기(사람별_점수):
    """사람별 점수를 평균 낸다. 이것이 점수판에 적는 값이다."""
    if len(사람별_점수) == 0:                          # 채점 대상이 없으면
        return 0.0                                     # 0 으로 본다
    return sum(사람별_점수.values()) / len(사람별_점수)


print("채점 함수 준비 완료 — 3주차와 재는 방식이 같습니다")
print(f"  학습 구간에 기록이 있는 사람 {len(이미_담은것)}명 · 전체 인기순위 {len(전체_인기순위)}개")

채점 함수 준비 완료 — 3주차와 재는 방식이 같습니다
  학습 구간에 기록이 있는 사람 285명 · 전체 인기순위 100개


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 사람 × 종목 표를 만든다</mark>

**4.2 에서 하는 일** — 거래 기록을 **표 한 장**으로 바꿉니다.

**1. 왜 표로 바꾸나**
- 거래 기록은 「누가 · 무엇을 · 언제」가 한 줄씩 적힌 **긴 목록**입니다. 모델은 이 모양을 그대로 못 읽습니다.
- 세로 **사람**, 가로 **종목**인 표로 바꿔 줘야 합니다. 담았으면 **1**, 안 담았으면 **0**.

**2. 이 표의 빈칸이 오늘의 과녁입니다**
- 칸이 285 × 100 = **28,500개**인데 1 이 들어가는 칸은 **7,095개**뿐입니다. 나머지 75%는 0 입니다.
- 이 **0 은 「싫다」가 아니라 「아직 모른다」**입니다. 그 빈칸을 그럴듯한 숫자로 채우는 것이 오늘 할 일입니다.

#### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 새로 나오는 문법 — <code>pd.crosstab</code></mark>

`pd.crosstab(세로, 가로)` — 두 열을 받아 **가로세로 표**를 만듭니다. 칸에는 **그 조합이 몇 번 나왔는지**가 들어갑니다.

`(표 > 0).astype(int)` — 0보다 크면 **1**, 아니면 **0** 으로 바꿉니다.
- `astype` 은 **타입을 바꾼다**는 뜻입니다. True/False 를 1/0 으로 바꿔 줍니다.
- 우리는 **몇 번 담았는지가 아니라 담았는지 아닌지**만 봅니다. 그래서 1과 0으로 눌러 줍니다.

아래 셀은 마음껏 고쳐 보셔도 됩니다.

In [5]:
# 이 셀에서 하는 일 — 작은 표로 crosstab 을 먼저 해 본다
# 왜 하나 — 8,086건에 그대로 쓰기 전에 무슨 일이 일어나는지 5줄로 확인한다
작은기록 = pd.DataFrame({                                # 손으로 만든 거래 기록 5줄
    "사람": ["가", "가", "나", "나", "다"],
    "종목": ["애플", "카카오", "애플", "애플", "네이버"],
})
print("작은 기록")                                        # 펼치기 전 모습
print(작은기록)

작은표 = pd.crosstab(작은기록["사람"], 작은기록["종목"])    # 사람 × 종목 표로 펼친다
print("\ncrosstab 으로 펼친 표 — 칸은 담은 횟수")
print(작은표)

작은표 = (작은표 > 0).astype(int)                         # 담았으면 1, 아니면 0
print("\n1 과 0 으로 누른 표 — 「나」가 애플을 두 번 담았어도 1 입니다")
print(작은표)

작은 기록
  사람   종목
0  가   애플
1  가  카카오
2  나   애플
3  나   애플
4  다  네이버

crosstab 으로 펼친 표 — 칸은 담은 횟수
종목  네이버  애플  카카오
사람              
가     0   1    1
나     0   2    0
다     1   0    0

1 과 0 으로 누른 표 — 「나」가 애플을 두 번 담았어도 1 입니다
종목  네이버  애플  카카오
사람              
가     0   1    1
나     0   1    0
다     1   0    0


In [6]:
# 이 셀에서 하는 일 — 진짜 데이터로 사람 × 종목 표를 만든다
# 왜 하나 — 이 표가 오늘 쪼갤 대상이다. 학습 구간만 쓴다 — 채점 구간을 보면 반칙이니까
표 = pd.crosstab(train["user_id"], train["item_id"])      # 학습 구간만으로 펼친다
표 = (표 > 0).astype(int)                                  # 담았으면 1, 아니면 0
표 = 표.reindex(columns=items["item_id"], fill_value=0)    # 아무도 안 담은 종목도 열로 세운다

칸_전체 = 표.shape[0] * 표.shape[1]                        # shape = (세로, 가로)
칸_채움 = int(표.values.sum())                             # 1 이 들어간 칸의 개수
print(f"표 크기 — 사람 {표.shape[0]}명 × 종목 {표.shape[1]}개 = 칸 {칸_전체:,}개")   # 슬라이드에서 본 28,500
print(f"1 인 칸 {칸_채움:,}개 = 전체의 {칸_채움 / 칸_전체:.1%}   ← 나머지는 전부 0")

print("\n왼쪽 위 귀퉁이만 잘라서 보기 (사람 5명 × 종목 6개)")
print(표.iloc[:5, :6])

표 크기 — 사람 285명 × 종목 100개 = 칸 28,500개
1 인 칸 7,095개 = 전체의 24.9%   ← 나머지는 전부 0

왼쪽 위 귀퉁이만 잘라서 보기 (사람 5명 × 종목 6개)
item_id  I001  I002  I003  I004  I005  I006
user_id                                    
U0001       0     0     0     0     0     1
U0002       0     1     0     0     0     0
U0003       0     0     0     0     1     0
U0004       1     0     1     1     1     0
U0005       0     0     0     0     0     0


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.3 큰 표를 작은 표 둘로 쪼갠다</mark>

**4.3 에서 하는 일** — 28,500칸짜리 표를 **사람표(285×8)** 와 **종목표(8×100)** 로 쪼갭니다.

**1. 쪼개면 무엇이 좋은가**
- 칸이 28,500개에서 **3,080개**로 줄어듭니다. 8개 안에 다 담으려면 **자잘한 것은 버리고 큰 취향만** 남겨야 합니다.
- 그 과정에서 **사람들 사이의 공통점**이 잡힙니다. 이것이 행렬분해가 하는 일의 전부입니다.

**2. 쪼갠 두 표를 다시 곱하면**
- 원래 크기(285×100)로 돌아오는데, **비어 있던 칸에도 숫자가 채워져 있습니다.**
- 그 숫자가 「이 사람이 이걸 담을 것 같다」는 **점수**입니다. 높은 순으로 10개를 주면 그게 추천입니다.

#### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 새로 나오는 문법 — <code>TruncatedSVD</code></mark>

```python
from sklearn.decomposition import TruncatedSVD
모델 = TruncatedSVD(n_components=8, random_state=42)
사람표 = 모델.fit_transform(표)
종목표 = 모델.components_
```

- `TruncatedSVD` — 행렬분해의 한 방법입니다. **Colab 에 이미 깔려 있어 설치가 없습니다.**
- `n_components=8` — **잠재 요인을 몇 개로 할지**입니다. 4.6 에서 이 숫자를 바꿔 봅니다.
- `fit_transform` — `fit`(표를 보고 쪼갤 방법을 정한다) + `transform`(그 방법으로 사람표를 만든다)을 한 번에 합니다.
- `components_` — 쪼개면서 같이 나온 **종목표**입니다. 뒤에 붙은 밑줄은 「학습하고 나서 생긴 값」이라는 표시입니다.
- `random_state=42` — 3주차와 같습니다. **누가 돌려도 같은 결과**가 나오게 고정합니다.

In [7]:
# 이 셀에서 하는 일 — 표를 쪼개고, 쪼갠 두 표의 크기와 한 사람의 숫자 8개를 본다
# 왜 하나 — 사람마다 숫자 8개를 갖게 된다는 것을 눈으로 확인한다
from sklearn.decomposition import TruncatedSVD     # 행렬분해 도구

모델 = TruncatedSVD(n_components=8, random_state=42)   # 잠재 요인 8개로 쪼갠다
사람표 = 모델.fit_transform(표)                         # 285 × 8 — 사람마다 숫자 8개
종목표 = 모델.components_                               # 8 × 100 — 종목마다 숫자 8개

print(f"쪼개기 전 — 사람 {표.shape[0]} × 종목 {표.shape[1]} = 칸 {표.shape[0] * 표.shape[1]:,}개")
print(f"쪼갠 뒤   — 사람표 {사람표.shape} + 종목표 {종목표.shape} = 칸 "
      f"{사람표.size + 종목표.size:,}개")

사람_이름들 = list(표.index)                            # 표의 세로 이름(사람) 목록
자리 = 사람_이름들.index("U0003")                       # U0003 이 몇 번째 줄인지
print(f"\nU0003 의 잠재 요인 8개 : {np.round(사람표[자리], 2)}")
print("  이 숫자 여덟 개에는 이름이 없습니다. 무슨 축인지는 슬라이드에서 열어 봤습니다.")   # 그래서 「잠재」다

쪼개기 전 — 사람 285 × 종목 100 = 칸 28,500개
쪼갠 뒤   — 사람표 (285, 8) + 종목표 (8, 100) = 칸 3,080개

U0003 의 잠재 요인 8개 : [ 3.16 -1.38  1.43  0.66 -1.09  0.05  0.09  0.28]
  이 숫자 여덟 개에는 이름이 없습니다. 무슨 축인지는 슬라이드에서 열어 봤습니다.


In [8]:
# 이 셀에서 하는 일 — 쪼갠 두 표를 다시 곱해 점수표를 만든다
# 왜 하나 — 비어 있던 칸에 숫자가 채워진 것을 직접 본다
점수표 = 사람표 @ 종목표                                 # @ = 행렬 곱하기. 285 × 100 으로 돌아온다
print("점수표 크기 :", 점수표.shape, "  ← 원래 표와 같은 크기")

열이름 = list(표.columns)                                # 표의 가로 이름(종목 번호) 목록
안_담은것 = []                                           # U0003 이 학습 구간에 안 담은 종목
for 번호 in 열이름:                                      # 종목을 하나씩
    if 표.loc["U0003", 번호] == 0:                       # 표에서 0 이면 안 담은 것
        안_담은것.append(번호)

안_담은것_점수 = {}                                      # {종목 번호: 점수표에 붙은 점수}
for 번호 in 안_담은것:                                    # 안 담은 종목을 하나씩
    안_담은것_점수[번호] = 점수표[자리][열이름.index(번호)]   # 그 칸의 점수를 꺼낸다

print(f"\nU0003 이 학습 구간에 안 담은 종목 {len(안_담은것)}개 — 표에서는 전부 0 이었습니다")
print("그런데 점수표에는 이렇게 숫자가 붙어 있습니다 (높은 순 세 개)")
for 번호 in sorted(안_담은것_점수, key=안_담은것_점수.get, reverse=True)[:3]:   # 큰 순으로 셋
    print(f"  {이름_사전[번호]:34} 점수 {안_담은것_점수[번호]:.3f}   ← 추천 후보")

점수표 크기 : (285, 100)   ← 원래 표와 같은 크기

U0003 이 학습 구간에 안 담은 종목 71개 — 표에서는 전부 0 이었습니다
그런데 점수표에는 이렇게 숫자가 붙어 있습니다 (높은 순 세 개)
  TIGER 미국나스닥100레버리지                 점수 0.641   ← 추천 후보
  Salesforce                         점수 0.605   ← 추천 후보
  iShares Global Clean Energy ETF    점수 0.603   ← 추천 후보


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.4 추천 10개를 고른다  ✏️ 직접 해 보기</mark>

**4.4 에서 하는 일** — 점수표에서 **높은 순으로 10개**를 뽑아 추천 목록을 만듭니다.

**1. 두 가지를 빼고 줍니다**
- **이미 담은 것**은 뺍니다. 이미 가진 것을 다시 추천하면 맞혀도 실력이 아닙니다(2주차에 정한 규칙).
- **학습 구간에 기록이 없는 사람**은 점수표에 줄이 없습니다. 이 사람들에게는 **전체 인기 목록**을 줍니다.

**2. 빈칸은 한 줄입니다**
- 점수가 높은 순서대로 **자리 번호**를 얻는 줄입니다. 바로 아래에서 작은 예제로 먼저 연습합니다.

#### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 새로 나오는 문법 — <code>np.argsort</code></mark>

`np.argsort(숫자묶음)` — **작은 순서대로** 그 값이 있던 **자리 번호**를 돌려줍니다. 값이 아니라 **자리**입니다.

- 큰 순서로 받고 싶으면 **앞에 마이너스**를 붙입니다 — `np.argsort(-숫자묶음)`.
- `arg` 는 argument(자리), `sort` 는 정렬입니다. **「정렬했을 때의 자리 번호」**라는 뜻입니다.

아래 셀은 마음껏 고쳐 보셔도 됩니다.

In [9]:
# 이 셀에서 하는 일 — 작은 숫자 다섯 개로 argsort 를 먼저 해 본다
# 왜 하나 — 값이 아니라 「자리 번호」가 나온다는 것을 확인하고 넘어간다
점수_다섯개 = np.array([0.1, 0.9, 0.4, 0.7, 0.2])       # 다섯 종목의 점수라고 치자
종목_다섯개 = ["가나다", "라마바", "사아자", "차카타", "파하가"]

print("점수      :", 점수_다섯개)                      # 값 자체
print("작은 순 자리 :", np.argsort(점수_다섯개))           # 가장 작은 0.1 이 0번 자리           # 가장 작은 0.1 이 0번 자리
print("큰 순 자리   :", np.argsort(-점수_다섯개))          # 마이너스를 붙이면 큰 순

print("\n큰 순으로 이름을 늘어놓으면")
for 자리번호 in np.argsort(-점수_다섯개):                  # 큰 순서대로 자리를 하나씩
    print(f"  {종목_다섯개[자리번호]} ({점수_다섯개[자리번호]})")

점수      : [0.1 0.9 0.4 0.7 0.2]
작은 순 자리 : [0 4 2 3 1]
큰 순 자리   : [1 3 2 4 0]

큰 순으로 이름을 늘어놓으면
  라마바 (0.9)
  차카타 (0.7)
  사아자 (0.4)
  파하가 (0.2)
  가나다 (0.1)


In [10]:
# 이 셀에서 하는 일 — ✏️ 빈칸 1 · 점수가 높은 순으로 추천 10개를 고른다
# 왜 하나 — 여기가 오늘 만드는 추천의 심장이다. 나머지는 전부 지난주 것이다
def 추천하기(사람):
    """그 사람의 점수표 한 줄을 보고 높은 순으로 10개를 돌려준다."""
    이미 = 이미_담은것.get(사람, set())                   # 학습 구간에 이미 담은 것
    if 사람 not in 사람_이름들:                            # 표에 줄이 없는 사람(신규)은
        return recsys.take(전체_인기순위, 이미)[:10]        # 전체 인기 목록으로 준다

    내_점수 = 점수표[사람_이름들.index(사람)]               # 그 사람 줄 = 종목 100개의 점수
    내_순서 = []                                          # 점수가 높은 종목부터 담을 목록
    for 자리번호 in np.argsort(-내_점수):                  # ← ✏️ 빈칸 : 높은 순 자리 번호
        내_순서.append(열이름[자리번호])                    # 자리 번호를 종목 번호로 바꿔 담는다
    return recsys.take(내_순서, 이미)[:10]                 # 이미 담은 것을 빼고 위에서 10개


내_추천 = 추천하기("U0003")                                # 함수를 만들었으면 한 명 돌려 본다
print("U0003 에게 줄 추천 10개")
for 순위, 번호 in enumerate(내_추천, start=1):             # enumerate = 번호를 붙여 준다
    print(f"  {순위:2}위  {이름_사전[번호]:34} 위험도 {위험도_사전[번호]}")

U0003 에게 줄 추천 10개
   1위  TIGER 미국나스닥100레버리지                 위험도 5
   2위  Salesforce                         위험도 4
   3위  iShares Global Clean Energy ETF    위험도 4
   4위  ProShares UltraPro Short QQQ       위험도 5
   5위  Vanguard FTSE Developed Markets ETF 위험도 3
   6위  LG에너지솔루션                           위험도 5
   7위  Global X Lithium & Battery Tech ETF 위험도 5
   8위  Vanguard S&P 500 ETF               위험도 3
   9위  TIGER 200                          위험도 3
  10위  알테오젠                               위험도 5


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.5 채점한다</mark>

**4.5 에서 하는 일** — 방금 만든 추천 방식을 **3주차 채점 함수**에 넣습니다.

**1. 무엇과 견주나**
- 2·3주차 점수는 **0.2684** 였습니다. 세그먼트 15개로 나눠 목록을 줬을 때의 점수입니다.
- 재는 방법은 **한 글자도 안 바꿨습니다.** 그러니 숫자가 달라졌다면 원인은 **추천 방식** 하나뿐입니다.

**2. 점수 말고 하나 더 봅니다**
- **서로 다른 목록이 몇 가지 나왔는지** 세어 봅니다. 2주차에는 15가지뿐이었습니다(세그먼트가 15개니까요).

In [11]:
# 이 셀에서 하는 일 — 방금 만든 추천 방식을 3주차 채점 함수로 잰다
# 왜 하나 — 잣대가 같아야 2·3주차 0.2684 와 견줄 수 있다
점수들 = 채점하기(추천하기)                                # {사람: Recall@10}
행렬분해_점수 = 평균내기(점수들)                            # 점수판에 적을 값

print(f"4주차 행렬분해 — Recall@10 = {행렬분해_점수:.4f}   (채점 대상 {len(점수들)}명)")   # 오늘 만든 점수
print(f"2·3주차 세그먼트 — Recall@10 = 0.2684")   # 견줄 상대
print()
오른폭 = (행렬분해_점수 - 0.2684) / 0.2684                  # (새 점수 - 옛 점수) ÷ 옛 점수
print(f"→ {오른폭:+.1%} 올랐습니다. 바뀐 것은 추천을 만드는 방법 하나입니다.")

4주차 행렬분해 — Recall@10 = 0.3918   (채점 대상 266명)
2·3주차 세그먼트 — Recall@10 = 0.2684

→ +46.0% 올랐습니다. 바뀐 것은 추천을 만드는 방법 하나입니다.


In [12]:
# 이 셀에서 하는 일 — 서로 다른 추천 목록이 몇 가지 나왔는지 센다
# 왜 하나 — 2주차에는 15가지뿐이었다. 개인화가 실제로 되는지는 이 숫자가 말해 준다
목록_모음 = set()                                          # 중복을 저절로 없애 주는 묶음
for 사람 in 점수들:                                        # 채점한 사람을 하나씩
    목록_모음.add(tuple(추천하기(사람)))                    # 목록을 통째로 넣는다(tuple 이라야 들어간다)

print(f"채점 대상 {len(점수들)}명에게 나간 목록 — 서로 다른 것이 {len(목록_모음)}가지")   # 개인화가 됐는지
print(f"2주차에는 세그먼트가 15개였으니 {15}가지였습니다.")
print()
print("→ 같은 세그먼트면 같은 목록을 받던 문제가 풀렸습니다.")

채점 대상 266명에게 나간 목록 — 서로 다른 것이 252가지
2주차에는 세그먼트가 15개였으니 15가지였습니다.

→ 같은 세그먼트면 같은 목록을 받던 문제가 풀렸습니다.


**결과 — 규칙을 안 정했더니 올랐습니다**

| 주차 | 규칙을 누가 정했나 | Recall@10 |
|---|---|---|
| 1주차 | 규칙이랄 것도 없음 (인기순) | 0.2163 |
| 2·3주차 | 사람 — 주력 섹터·평균 위험도 | 0.2684 |
| **4주차** | **아무도 안 정함 — 모델이 찾음** | **0.3918** |

**바뀐 것은 추천을 만드는 방법 하나뿐입니다.** 채점은 3주차 함수를 그대로 썼습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.6 요인 개수를 바꿔 본다  ✏️ 직접 해 보기</mark>

**4.6 에서 하는 일** — 잠재 요인 개수만 바꿔 **다섯 번** 돌려 보고 점수를 견줍니다.

**1. 왜 바꿔 보나**
- 8개는 **정답이 아니라 골라 본 값**입니다. 많이 주면 좋아질 것 같지만 그렇지 않습니다.
- **너무 적으면** 취향을 두세 덩어리로만 나눕니다. **너무 많으면** 한 번 담아 본 것까지 외워 버립니다(**과적합**).

**2. 무엇을 채우나**
- 돌려 볼 숫자 목록 한 줄입니다. **2, 4, 8, 16, 32** 를 넣어 보세요. 한 번 도는 데 1초도 안 걸립니다.

In [13]:
# 이 셀에서 하는 일 — ✏️ 빈칸 2 · 요인 개수만 바꿔 다섯 번 돌린다
# 왜 하나 — 「많을수록 좋다」가 아니라는 것을 점수로 확인한다
해볼_개수들 = [2, 4, 8, 16, 32]        # ← ✏️ 빈칸 : 돌려 볼 요인 개수를 넣어 보세요

print("요인 개수   Recall@10")
결과_모음 = {}                                              # {요인 개수: 점수}
for 개수 in 해볼_개수들:                                    # 개수를 하나씩 바꿔 가며
    그때_모델 = TruncatedSVD(n_components=개수, random_state=42)   # 그 개수로 쪼갠다
    그때_사람표 = 그때_모델.fit_transform(표)                # 285 × 개수
    그때_점수표 = 그때_사람표 @ 그때_모델.components_         # 다시 곱해 점수표

    def 그때_추천하기(사람, 점수표=그때_점수표):              # 위 추천 함수와 같은 방식
        이미 = 이미_담은것.get(사람, set())
        if 사람 not in 사람_이름들:
            return recsys.take(전체_인기순위, 이미)[:10]
        내_순서 = []
        for 자리번호 in np.argsort(-점수표[사람_이름들.index(사람)]):
            내_순서.append(열이름[자리번호])
        return recsys.take(내_순서, 이미)[:10]

    결과_모음[개수] = 평균내기(채점하기(그때_추천하기))        # 같은 채점 함수로 잰다
    print(f"  {개수:>5}개   {결과_모음[개수]:.4f}")

가장_높은_개수 = max(결과_모음, key=결과_모음.get)           # 점수가 가장 높은 개수
print()
print(f"→ 가장 높은 것은 요인 {가장_높은_개수}개 · {결과_모음[가장_높은_개수]:.4f} 입니다.")
print(f"→ 요인을 {해볼_개수들[-1]}개까지 늘리면 {결과_모음[해볼_개수들[-1]]:.4f} 로 오히려 떨어집니다.")

요인 개수   Recall@10
      2개   0.2760
      4개   0.3452
      8개   0.3918
     16개   0.3475
     32개   0.2985

→ 가장 높은 것은 요인 8개 · 0.3918 입니다.
→ 요인을 32개까지 늘리면 0.2985 로 오히려 떨어집니다.


**결과 — 산 모양이 나옵니다**

요인을 **네 배로 늘렸는데 점수는 떨어졌습니다.** 이것이 **과적합**입니다.

축이 많아지면 그 사람이 한 번 담아 본 것까지 전부 외울 수 있게 됩니다. 그런데 **외운 것으로는 안 본 것을 못 맞힙니다.**

그래서 요인 개수는 정답이 정해져 있지 않습니다. **데이터가 바뀌면 가장 높은 자리도 옮겨 갑니다.** 매번 돌려 보고 정합니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.7 기존과 신규를 갈라서 본다</mark>

**4.7 에서 하는 일** — 3주차와 똑같이 **기존 투자자**와 **신규 투자자**로 갈라 평균을 따로 냅니다.

**1. 왜 또 갈라 보나**
- 3주차에 신규 15명이 **0.1653** 이었습니다. 전체 평균 뒤에 숨어 있던 숫자였죠.
- 모델을 썼으니 이 사람들도 좋아졌을까요. **확인해 봐야 압니다.**

**2. 새로 채점하지 않습니다**
- 4.5 에서 받아 둔 사람별 점수 266개를 **두 바구니에 나눠 담고** 평균만 따로 냅니다.

In [14]:
# 이 셀에서 하는 일 — 채점 대상을 기존·신규로 갈라 평균을 따로 낸다
# 왜 하나 — 전체 평균 하나가 무엇을 가리고 있는지 보려는 것이다
신규_투자자 = set(test["user_id"]) - set(train["user_id"])   # 빼기(-) = 채점에만 있는 사람
print(f"신규 투자자 {len(신규_투자자)}명 — 학습 구간에 기록이 0건이라 표에 줄이 없습니다")   # 3주차와 같은 15명

기존_점수들 = {}                                            # {기존 투자자: 점수}
신규_점수들 = {}                                            # {신규 투자자: 점수}
for 사람, 점수 in 점수들.items():                            # 4.5 에서 받아 둔 점수를 하나씩
    if 사람 in 신규_투자자:                                  # 신규 명단에 있으면
        신규_점수들[사람] = 점수                              # 신규 바구니에
    else:                                                    # 아니면
        기존_점수들[사람] = 점수                              # 기존 바구니에

print()
print(f"전체 {len(점수들)}명  {평균내기(점수들):.4f}   (3주차 0.2684)")   # 괄호 안이 3주차 값
print(f"기존 {len(기존_점수들)}명  {평균내기(기존_점수들):.4f}   (3주차 0.2746)")
print(f"신규  {len(신규_점수들)}명  {평균내기(신규_점수들):.4f}   (3주차 0.1653)")

신규 투자자 15명 — 학습 구간에 기록이 0건이라 표에 줄이 없습니다

전체 266명  0.3918   (3주차 0.2684)
기존 251명  0.4054   (3주차 0.2746)
신규  15명  0.1653   (3주차 0.1653)


**결과 — 모델이 못 푸는 문제가 있습니다**

기존 투자자는 크게 올랐는데 **신규 15명은 3주차와 소수점 끝자리까지 같습니다.**

이 사람들은 학습 구간 기록이 0건이라 **사람 × 종목 표에 줄 자체가 없습니다.** 줄이 없으니 요인 8개도 안 나오고, 모델이 점수를 매길 방법이 없습니다. 그래서 3주차처럼 **전체 인기 목록**을 줬고, 같은 목록을 줬으니 같은 점수가 나왔습니다.

**모델이 좋아진다고 모든 문제가 풀리지는 않습니다.** 재료가 없는 사람에게는 어떤 모델도 소용이 없습니다. 금융 앱이 가입 직후에 투자 성향 설문을 받는 이유가 이것입니다 — **재료를 만들어 내는 것**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.8 요인 개수를 몇으로 정할지 적는다  ✏️ 직접 해 보기</mark>

**4.8 에서 하는 일** — 서비스에 올린다면 요인을 몇 개로 할지 정하고, **왜 그런지 한 줄** 적습니다.

- 점수가 가장 높은 값이 정답처럼 보이지만, **데이터가 늘면 그 자리는 옮겨 갑니다.**
- 실무에서 하는 일이 정확히 이것입니다 — 숫자 하나로 안 정해지는 것을 **판단하고 근거를 남기는 일**.

In [15]:
# 이 셀에서 하는 일 — ✏️ 내가 고른 요인 개수와 그 이유를 적는다
# 왜 하나 — 오늘의 과제는 점수가 아니라 「왜 그렇게 정했는지 말할 수 있는 것」이다
내가_고른_개수 = 8                      # ← ✏️ 바꿔도 됩니다
내_이유 = "점수가 가장 높았고, 더 늘리면 외우기 시작해서"   # ← ✏️ 한 줄로 적어 주세요

print(f"내가 고른 요인 개수 : {내가_고른_개수}개")        # 위에서 적은 값을 확인
print(f"고른 이유           : {내_이유}")
if 내가_고른_개수 in 결과_모음:                        # 4.6 에서 돌려 본 개수라면
    print(f"그때의 점수         : {결과_모음[내가_고른_개수]:.4f}")
else:                                                  # 안 돌려 본 개수를 적었다면
    print("그때의 점수         : 4.6 에서 돌려 보지 않은 개수입니다. 목록에 넣고 다시 돌려 보세요.")

내가 고른 요인 개수 : 8개
고른 이유           : 점수가 가장 높았고, 더 늘리면 외우기 시작해서
그때의 점수         : 0.3918


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.9 오늘의 마무리  `[PPT]`</mark>

오늘은 **규칙을 사람이 정하지 않았습니다.** 거래 기록만 주고 모델이 찾게 했더니 점수가 크게 올랐습니다.

1. **0.2684 → 0.3918** — 10주 중 가장 많이 오른 주입니다.
2. **목록이 15가지에서 252가지로** — 이제 사람마다 다릅니다. 2주차에 남겨 둔 숙제가 풀렸습니다.
3. **신규 15명은 그대로** — 재료가 없으면 모델도 못 합니다.

다음 주에는 종목을 **전부 재지 않고** 후보를 먼저 추립니다. 점수가 아니라 **규모가 커져도 버티는 구조**를 얻는 주입니다.

In [16]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 4주차는 10주 중 가장 크게 오른 줄이다. 다음 주부터는 이 값과 견준다
recsys.record(4, "행렬분해 SVD · 요인 8개", 행렬분해_점수,
              note="규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다")

print("점수판에 4주차를 기록했습니다.\n")          # \n = 한 줄 띄우기
recsys.leaderboard(upto=4)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 4 · 행렬분해 SVD · 요인 8개 · Recall@10 = 0.3918
점수판에 4주차를 기록했습니다.



,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
